# 11 · AWS Glue: portable checks and native DQDL

Run this optional notebook in an AWS Glue Spark interactive session after uploading the supplied `data/raw` folder to S3. This notebook uses Glue libraries; it is not expected to run on plain Spark. No crawler, Hive metastore or Glue Data Catalog table is required. Use a Glue execution role with access to your training prefix.


## Glue session configuration
Choose a Spark 3.5-based Glue runtime in the AWS console. Before starting the interactive session, configure the execution role and region using your organization’s settings. Native Glue DQ results are dataset/rule results; they do not automatically implement our custom quarantine metadata. S3 and Glue session usage incur AWS charges.


In [ ]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsgluedq.transforms import EvaluateDataQuality

glueContext = GlueContext(SparkContext.getOrCreate())
spark = glueContext.spark_session


## Storage and cast policy
Use the same S3 prefix as notebook 00. Preserve raw values when converting. No access keys are embedded.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)

assert BASE_PATH.startswith(("s3://", "s3a://")), "Use an S3 prefix in Glue"
source = spark.read.json(f"{RAW_PATH}/orders")
df = (
    source.withColumn("quantity_raw", F.col("quantity"))
    .withColumn("quantity", F.expr("try_cast(quantity as int)"))
    .withColumn("status", F.upper(F.trim("status")))
)
df.show(30, truncate=False)


## Evaluate native Glue rules
Map completeness, uniqueness, ranges and domains to DQDL. This sample intentionally fails several rules. Publishing to CloudWatch/Glue results is disabled by default; switch it on only when the execution role and reporting destination are configured. Evaluation alone does not stop a job.


In [ ]:
ruleset = """Rules = [
    IsComplete "customer_id",
    IsUnique "order_id",
    ColumnValues "quantity" > 0,
    ColumnValues "status" in ["CREATED", "PAID", "SHIPPED", "CANCELLED"],
    RowCount > 0
]"""
dynamic = DynamicFrame.fromDF(df, glueContext, "orders")
results = EvaluateDataQuality.apply(
    frame=dynamic,
    ruleset=ruleset,
    publishing_options={
        "dataQualityEvaluationContext": "order-quality_orders",
        "enableDataQualityCloudWatchMetrics": False,
        "enableDataQualityResultsPublishing": False,
    },
)
rule_results = results.toDF()
rule_results.show(truncate=False)
rule_results.write.mode("errorifexists").parquet(f"{AUDIT_PATH}/glue_native/{RUN_ID}")
failed_rules = rule_results.filter(F.col("Outcome") != "Passed").count()
print("Publication gate:", "FAIL" if failed_rules else "PASS")
FAIL_JOB_ON_GATE = (
    False  # True for an actual scheduled job; False to inspect this lesson
)
if FAIL_JOB_ON_GATE and failed_rules:
    raise RuntimeError(f"Glue DQ gate failed: {failed_rules} rules; run={RUN_ID}")


## Compare contracts rather than names
IsComplete detects nulls, not every whitespace-only business value. Explicit normalization and custom predicates still matter. Native DQDL rules have documented null and type semantics; compare them against notebook 03 before substituting them. Run notebook 10 directly in a Glue Spark session for the full portable quarantine/reconciliation pipeline.

[Glue EvaluateDataQuality reference](https://docs.aws.amazon.com/glue/latest/dg/aws-glue-api-crawler-pyspark-transforms-EvaluateDataQuality.html) · [DQDL reference](https://docs.aws.amazon.com/glue/latest/dg/dqdl.html).
